# Prompt Testing and Iteration

When a prompt does not behave the way you want, it is tempting to keep tweaking it and rerunning a single example until it looks right. That approach hides problems: a change that fixes one case often quietly breaks another.

This notebook takes a more reliable approach: you build a small set of test cases, run each prompt variant against all of them, and compare the results, so you can see what a change actually does before you commit to it. You will follow this loop throughout:

```mermaid
flowchart LR
    A[Task] --> B[Test cases]
    B --> C[Prompt variant]
    C --> D[Run chain]
    D --> E[Inspect outputs]
    E --> F[Revise prompt]
    F --> C
```

## Learning Objectives

By the end of this notebook, you should be able to:

- Build a small evaluation set that exposes a prompt's weak spots.
- Compare a baseline prompt against a stricter one on the same cases.
- Apply few-shot examples to improve consistency on harder cases.
- Reuse this prompt-testing loop in your own projects.

In [ ]:
# Keep the setup cell focused so the notebook is easy to rerun from the top.
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

from llm_config import create_chat_model


def print_outputs(cases: list[dict[str, str]], outputs: list[str]) -> None:
    for case, output in zip(cases, outputs):
        print("=" * 100)
        print(f"Case: {case['label']}")
        print(f"What to watch: {case['watch_for']}")
        print(output)

In [ ]:
load_dotenv()

llm = create_chat_model(temperature=0.1, max_tokens=512)

output_parser = StrOutputParser()

## Step 1 - Create a Tiny Evaluation Set

For prompt testing, a tiny but varied test set is often more useful than one perfect example. The goal is to include a few cases that are different enough to expose prompt weaknesses.

In this notebook, we will use a simple support-triage task. Each case is a short customer message, and we want the model to:
- estimate urgency
- identify the topic
- suggest a short response


In [ ]:
# Keep the test set small enough to inspect manually.
support_cases = [
    {
        "label": "Damaged package",
        "message": "My package arrived today and the mug inside was broken. I need a replacement before Friday because it is a gift.",
        "watch_for": "Should likely be high urgency and mention replacement or support steps.",
    },
    {
        "label": "Password reset",
        "message": "I cannot sign in to my account because I forgot my password. How do I reset it?",
        "watch_for": "Should be account-related and not overstate urgency.",
    },
    {
        "label": "Double charge",
        "message": "I think my card was charged twice for the same order. Can someone check this?",
        "watch_for": "Should identify billing and treat it as fairly urgent.",
    },
    {
        "label": "Sustainability question",
        "message": "Your website says your products are sustainable. What materials do you actually use?",
        "watch_for": "Should stay low urgency and answer as a product question, not a complaint.",
    },
]

support_cases

A compact evaluation set like this helps you ask practical questions:
- Which cases produce messy or inconsistent output?
- Which cases are misclassified?
- Which instructions are too vague?

Notice that the test set mixes logistics, account access, billing, and product questions. That variety makes the prompt easier to stress-test.

## Step 2 - Start With a Baseline Prompt

We begin with a deliberately simple prompt. It is good enough to run, but it leaves a lot of room for the model to choose its own output structure.

In [ ]:
# This baseline prompt defines the task but does not strongly constrain the format.
baseline_template = """You are helping an ecommerce support team.
Read the customer message and identify the urgency, the topic, and a short reply.

Customer message:
{message}
"""

baseline_prompt = PromptTemplate(
    template=baseline_template,
    input_variables=["message"],
)

baseline_chain = baseline_prompt | llm | output_parser

In [ ]:
# Try the baseline prompt on one case before scaling up.
print(baseline_chain.invoke({"message": support_cases[0]["message"]}))

## Step 3 - Evaluate the Baseline Across All Cases

One output is not enough to judge a prompt. Prompt testing becomes more informative when you run the same prompt on several cases and compare the outputs side by side.

In [ ]:
# batch() lets us reuse the same prompt on many cases at once.
baseline_inputs = [{"message": case["message"]} for case in support_cases]
baseline_outputs = baseline_chain.batch(baseline_inputs)
print_outputs(support_cases, baseline_outputs)

### What Usually Goes Wrong in the Baseline

With a loose prompt, common issues include:
- inconsistent formatting from one case to another
- uneven detail across cases
- vague urgency labels
- responses that are friendly but hard to compare systematically

This is a good sign that we need a stronger prompt specification, not necessarily a different model.

## Step 4 - Improve the Prompt With Stronger Instructions

Now we make the task more explicit. We still use the same model, but we add tighter rules for the output format so the results become easier to inspect and compare.

In [ ]:
# Stronger instructions often improve consistency more than changing the model does.
improved_template = """You are triaging ecommerce support messages.
Analyze the customer message and return exactly this format:

Urgency: <low|medium|high>
Topic: <shipping|returns|billing|account|product>
Suggested reply:
- sentence 1
- sentence 2

Keep the reply concise and practical.

Customer message:
{message}
"""

improved_prompt = PromptTemplate(
    template=improved_template,
    input_variables=["message"],
)

improved_chain = improved_prompt | llm | output_parser

In [ ]:
# Compare the improved prompt on the same evaluation set.
improved_outputs = improved_chain.batch(baseline_inputs)
print_outputs(support_cases, improved_outputs)

#### Compare Two Prompt Variants on One Hard Case

Comparing full test sets is useful, but sometimes you also want to zoom in on one case that feels tricky and inspect how each prompt handles it.

In [ ]:
hard_case = support_cases[2]["message"]

print("BASELINE")
print(baseline_chain.invoke({"message": hard_case}))
print("\n" + "-" * 100 + "\n")
print("IMPROVED")
print(improved_chain.invoke({"message": hard_case}))

## Step 5 - Add Few-Shot Examples for Harder Edge Cases

If the output is still inconsistent, few-shot examples can teach the model the exact style you want. This is especially useful when the format matters or when the task includes edge cases that the model may interpret differently.

```mermaid
flowchart LR
    A[Example input-output pairs] --> B[FewShotPromptTemplate]
    C[New customer message] --> B
    B --> D[Formatted prompt]
    D --> E[More controlled output]
```

The examples do not retrain the model. They just give the model a clearer pattern to imitate for the current request.

In [ ]:
# Each example demonstrates the output style we want on a realistic support case.
examples = [
    {
        "message": "My order still has not arrived and I need it tomorrow for an event.",
        "analysis": "Urgency: high\nTopic: shipping\nSuggested reply:\n- I am sorry your order has not arrived in time for your event.\n- Please send us your order number so we can check shipping options right away.",
    },
    {
        "message": "I forgot my password and cannot get into my account.",
        "analysis": "Urgency: medium\nTopic: account\nSuggested reply:\n- Please use the password reset link on the sign-in page to create a new password.\n- If that does not work, contact support and we can help verify your account.",
    },
]

example_prompt = PromptTemplate(
    input_variables=["message", "analysis"],
    template="""Example message:\n{message}\n\nExample output:\n{analysis}""",
)

prefix = """You are triaging ecommerce support messages.
Return exactly this format:
Urgency: <low|medium|high>
Topic: <shipping|returns|billing|account|product>
Suggested reply:
- sentence 1
- sentence 2

Use the examples below as style guidance."""

suffix = """Now analyze this customer message:
{message}"""

few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix=prefix,
    suffix=suffix,
    input_variables=["message"],
    example_separator="\n\n",
)

few_shot_chain = few_shot_prompt | llm | output_parser

#### Inspect the Final Few-Shot Prompt

When working with more complex prompt templates, printing the final prompt is a great debugging habit. It lets you check whether the examples and the new input were assembled the way you expected.

In [ ]:
print(few_shot_prompt.format(message=support_cases[0]["message"]))

In [ ]:
# Evaluate the few-shot prompt on the same test set.
few_shot_outputs = few_shot_chain.batch(baseline_inputs)
print_outputs(support_cases, few_shot_outputs)

## Step 6 - Reuse the Prompt on a New Test Set

A prompt is more convincing when it generalizes beyond the first set of examples. Here we apply the same few-shot prompt to a second mini test set with different customer messages.

In [ ]:
new_cases = [
    {
        "label": "Gift card request",
        "message": "Do you sell gift cards? I want to send one to a friend for their birthday.",
        "watch_for": "Should be product-related and low urgency.",
    },
    {
        "label": "Express shipping",
        "message": "I placed my order an hour ago. Can I upgrade it to express shipping?",
        "watch_for": "Should likely be shipping-related and time-sensitive.",
    },
]

new_outputs = few_shot_chain.batch([{"message": case["message"]} for case in new_cases])
print_outputs(new_cases, new_outputs)

## Exercises

1. Add one more support case to `support_cases` and test all three prompt variants.
2. Change the improved prompt so it must return JSON-like output instead of bullet points.
3. Add one more few-shot example for a returns-related message and compare the result.
4. Raise the model temperature and check whether output consistency gets better or worse.


## My Exercise Answers

The cells below answer the four exercises in order. Each one reuses the chains and test cases defined above, so run the notebook from the top first.

### Exercise 1 - One More Support Case, All Three Prompt Variants

The new case is an address change on an order that ships tomorrow. It is a useful stress test because the customer is polite and calm, but the request is time-critical: a prompt that judges urgency by tone instead of by deadline will rate it too low.

In [ ]:
# Add the new case once, even if this cell is run again.
address_case = {
    "label": "Address change",
    "message": "I just noticed my order is going to my old address. It ships tomorrow morning. Can you change the delivery address before it leaves?",
    "watch_for": "Should be shipping and high urgency: calm tone, but a hard deadline.",
}

if address_case not in support_cases:
    support_cases.append(address_case)

# Rebuild the inputs so every variant sees the same five cases.
all_inputs = [{"message": case["message"]} for case in support_cases]

variants = {
    "BASELINE": baseline_chain,
    "IMPROVED": improved_chain,
    "FEW-SHOT": few_shot_chain,
}

variant_outputs = {}
for name, chain in variants.items():
    variant_outputs[name] = chain.batch(all_inputs, config={"max_concurrency": 2})
    print("\n" + "#" * 40, name, "#" * 40)
    print_outputs(support_cases, variant_outputs[name])

Reading three long printouts side by side is slow, so the next cell pulls the `Urgency:` and `Topic:` lines out of each output and puts them in one small table. The baseline prompt has no fixed format, so its labels often cannot be extracted at all - which is itself the finding.

In [ ]:
import re


def extract_label(output: str, field: str) -> str:
    """Return the value after 'Urgency:' or 'Topic:', or '?' if the format is missing."""
    match = re.search(rf"^\W*{field}\W*:\W*(\w+)", output, flags=re.IGNORECASE | re.MULTILINE)
    return match.group(1).lower() if match else "?"


print(f"{'Case':<26}" + "".join(f"{name:<22}" for name in variants))
for i, case in enumerate(support_cases):
    row = f"{case['label']:<26}"
    for name in variants:
        output = variant_outputs[name][i]
        row += f"{extract_label(output, 'Urgency') + ' / ' + extract_label(output, 'Topic'):<22}"
    print(row)

### Exercise 2 - JSON Output Instead of Bullet Points

The improved prompt now asks for one JSON object. Two details matter:

- Literal braces in a `PromptTemplate` must be doubled (`{{` and `}}`), otherwise LangChain reads them as input variables. `{message}` stays the only real variable.
- `JsonOutputParser` replaces `StrOutputParser`, so the chain returns a Python dictionary instead of text. That turns "the format looks right" into something code can check.

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

json_template = """You are triaging ecommerce support messages.
Analyze the customer message and return only one JSON object, with no text before or after it.
Use exactly these keys and allowed values:

{{
  "urgency": "low" | "medium" | "high",
  "topic": "shipping" | "returns" | "billing" | "account" | "product",
  "suggested_reply": ["sentence 1", "sentence 2"]
}}

Keep the reply concise and practical.

Customer message:
{message}
"""

json_prompt = PromptTemplate(
    template=json_template,
    input_variables=["message"],
)

json_chain = json_prompt | llm | JsonOutputParser()

json_outputs = json_chain.batch(all_inputs, config={"max_concurrency": 2})

for case, result in zip(support_cases, json_outputs):
    print("=" * 100)
    print(f"Case: {case['label']}")
    print(result)

Because the output is now data, the format rules can be tested automatically instead of by eye:

In [ ]:
ALLOWED_URGENCY = {"low", "medium", "high"}
ALLOWED_TOPIC = {"shipping", "returns", "billing", "account", "product"}


def check_triage(result: dict) -> list[str]:
    """Return a list of format problems; an empty list means the output is valid."""
    problems = []
    if set(result) != {"urgency", "topic", "suggested_reply"}:
        problems.append(f"unexpected keys: {sorted(result)}")
    if result.get("urgency") not in ALLOWED_URGENCY:
        problems.append(f"urgency not allowed: {result.get('urgency')!r}")
    if result.get("topic") not in ALLOWED_TOPIC:
        problems.append(f"topic not allowed: {result.get('topic')!r}")
    reply = result.get("suggested_reply")
    if not isinstance(reply, list) or len(reply) != 2:
        problems.append("suggested_reply is not a list of 2 sentences")
    return problems


for case, result in zip(support_cases, json_outputs):
    problems = check_triage(result)
    status = "OK" if not problems else "PROBLEM: " + "; ".join(problems)
    print(f"{case['label']:<26}{result.get('urgency', '?'):<8}{result.get('topic', '?'):<10}{status}")

### Exercise 3 - A Returns Example for the Few-Shot Prompt

The two original examples cover `shipping` and `account` only. A third example now shows a returns message. To compare fairly, the old and the new few-shot prompt run on the same three returns-flavoured messages, including two where returns competes with another topic (a damaged delivery, a refund that has not been paid).

In [ ]:
returns_example = {
    "message": "The jacket I ordered is too small. How can I send it back?",
    "analysis": "Urgency: low\nTopic: returns\nSuggested reply:\n- You can return the jacket within 30 days as long as it is unworn and in its original packaging.\n- Start the return from your account dashboard and we will email you a prepaid label.",
}

few_shot_prompt_v2 = FewShotPromptTemplate(
    examples=examples + [returns_example],
    example_prompt=example_prompt,
    prefix=prefix,
    suffix=suffix,
    input_variables=["message"],
    example_separator="\n\n",
)

few_shot_chain_v2 = few_shot_prompt_v2 | llm | output_parser

returns_cases = [
    {
        "label": "Late refund",
        "message": "I returned my shoes last week but I still have not received the refund.",
        "watch_for": "Returns or billing? The return is done, the money is missing.",
    },
    {
        "label": "Damaged, wants refund",
        "message": "My package was damaged in transit and I want to send it back for a refund.",
        "watch_for": "Returns or shipping? The cause is shipping, the request is a return.",
    },
    {
        "label": "Plain return",
        "message": "These headphones are not what I expected. What is the return process?",
        "watch_for": "Clear returns case, low urgency.",
    },
]
returns_inputs = [{"message": case["message"]} for case in returns_cases]

outputs_two = few_shot_chain.batch(returns_inputs, config={"max_concurrency": 2})
outputs_three = few_shot_chain_v2.batch(returns_inputs, config={"max_concurrency": 2})

print("#" * 30, "TWO EXAMPLES (original)", "#" * 30)
print_outputs(returns_cases, outputs_two)
print("\n" + "#" * 30, "THREE EXAMPLES (with returns)", "#" * 30)
print_outputs(returns_cases, outputs_three)

print(f"\n{'Case':<26}{'two examples':<22}{'three examples':<22}")
for case, two, three in zip(returns_cases, outputs_two, outputs_three):
    label_two = extract_label(two, "Urgency") + " / " + extract_label(two, "Topic")
    label_three = extract_label(three, "Urgency") + " / " + extract_label(three, "Topic")
    print(f"{case['label']:<26}{label_two:<22}{label_three:<22}")

### Exercise 4 - Raise the Temperature

One run per temperature proves nothing, because a single output can look fine by chance. So the same message goes through the same improved prompt three times at temperature `0.1` and three times at `1.0`. Consistency is then counted: how many different labels and how many different full texts came back from three identical requests?

In [ ]:
RUNS = 3
temperature_message = support_cases[2]["message"]  # the double-charge case

temperature_results = {}
for temperature in (0.1, 1.0):
    test_llm = create_chat_model(temperature=temperature, max_tokens=512)
    test_chain = improved_prompt | test_llm | output_parser
    temperature_results[temperature] = [
        test_chain.invoke({"message": temperature_message}) for _ in range(RUNS)
    ]

for temperature, outputs in temperature_results.items():
    print("\n" + "#" * 30, f"temperature = {temperature}", "#" * 30)
    for run, output in enumerate(outputs, start=1):
        print(f"--- run {run} ---")
        print(output)

print(f"\n{'temperature':<14}{'different labels':<20}{'different full texts':<22}")
for temperature, outputs in temperature_results.items():
    labels = {(extract_label(o, "Urgency"), extract_label(o, "Topic")) for o in outputs}
    texts = {o.strip() for o in outputs}
    print(f"{temperature:<14}{len(labels):<20}{len(texts):<22}")

## Conclusion

In this notebook, you practiced a simple but powerful prompt-engineering workflow:
- define a task clearly
- create a compact evaluation set
- compare prompt variants on the same cases
- tighten the prompt format when outputs are inconsistent
- add few-shot examples when you need even more control

This kind of lightweight testing loop is a strong next step after learning the LangChain basics and the core prompt-engineering patterns.